In [ ]:
import torch
import torch.utils.data as data
from data.dataset import UltrasoundDataset # 导入我们刚才重构好的干净 Dataset

def test_my_dataset():
    # 填入你刚才提供的真实路径
    dataroot = '/home/liujia/g_linux/Simu_data_1channel/'
    dir_lq = 'Recon_SQ_75'   # 对应代码里的低质量/少角度输入
    dir_sq = 'Recon_HQ_33'   # 对应代码里的高质量 Ground Truth
    
    print(">>> 正在初始化 Dataset...")
    try:
        # 实例化 Dataset
        dataset = UltrasoundDataset(
            dataroot=dataroot,
            dir_lq=dir_lq,
            dir_sq=dir_sq,
            patch_size=(256, 64, 64),
            norm_min=-1.0, 
            norm_max=1.0
        )
        
        print(f">>> 数据集初始化成功！总共扫描到 {len(dataset)} 对 NIfTI 数据。")
        
        if len(dataset) == 0:
            print(">>> 警告：没有找到任何数据，请检查路径。")
            return

        # 抓取第一个数据测试裁剪和归一化
        print("\n>>> 正在随机裁剪第一个 Patch 进行测试...")
        sample = dataset[0]
        
        print(f"提取到的字典 Keys: {list(sample.keys())}")
        print(f"当前文件名 (case_name): {sample['case_name']}")
        print(f"75角度输入 Tensor 形状: {sample['lq'].shape} (物理期望: [1, 256, 64, 64])")
        print(f"33角度 GT Tensor 形状: {sample['hq'].shape} (物理期望: [1, 256, 64, 64])")
        
        print(f"75角度输入 最大值: {sample['lq'].max():.4f}, 最小值: {sample['lq'].min():.4f}")
        print(f"33角度 GT 最大值: {sample['hq'].max():.4f}, 最小值: {sample['hq'].min():.4f}")

        # 测试 DataLoader 的 Batch 拼装功能
        print("\n>>> 正在测试 PyTorch DataLoader 封装...")
        dataloader = data.DataLoader(dataset, batch_size=2, shuffle=True, num_workers=0)
        batch = next(iter(dataloader))
        print(f"Batch Tensor 形状: {batch['lq'].shape} (物理期望: [2, 1, 256, 64, 64])")
        print("\n>>> 测试结束。如果以上无报错，说明数据流模块重构完美成功！")

    except Exception as e:
        print(f"\n>>> 测试失败，出现报错: {e}")

if __name__ == '__main__':
    test_my_dataset()

In [ ]:
import torch
from networks.generator import AnisotropicUNet

def test_my_generator():
    print(">>> 正在初始化极简各向异性 Generator...")
    # 初始化网络
    model = AnisotropicUNet(input_nc=1, output_nc=1, ngf=64)
    
    # 我们直接用 torch.randn 模拟一个 DataLoader 喂过来的假数据
    # 模拟 BatchSize=2, Channel=1, Z=256, Y=64, X=64
    dummy_input = torch.randn(2, 1, 256, 64, 64)
    print(f">>> 模拟输入 Tensor 形状: {dummy_input.shape}")
    
    print(">>> 正在进行 U-Net 腹部前向传播 (Forward)...")
    # 不计算梯度，仅测试结构
    with torch.no_grad():
        output = model(dummy_input)
    
    print(f">>> 最终输出 Tensor 形状: {output.shape}")
    
    if output.shape == dummy_input.shape:
        print("\n>>> 测试完美通过！网络前向传播成功，输入输出维度严丝合缝！")
        print(">>> U-Net 的各向异性拼接逻辑极其健康！")

if __name__ == '__main__':
    test_my_generator()

In [ ]:
import torch
from networks.discriminator import Discriminator3D

print(">>> 正在初始化 3D 判别器...")
# 假设我们用 Conditional GAN 的方式，把 33角度输入 和 75角度目标/生成图 拼在一起送进去
# 拼接后通道数为 2，所以 input_nc=2
netD = Discriminator3D(input_nc=2, ndf=64)

# 模拟输入 [Batch=2, Channel=2, Z=256, Y=64, X=64]
dummy_input = torch.randn(2, 2, 256, 64, 64)
print(f">>> 模拟输入 Tensor 形状: {dummy_input.shape}")

output = netD(dummy_input)
print(f">>> 判别器输出 Tensor 形状: {output.shape}")
print(">>> 如果没有报错，且输出形状类似 [2, 1, 30, 6, 6]，说明判别器搭建完美！")

In [20]:
!python train.py \
  --dataroot /home/liujia/g_linux/Simu_data_1channel/ \
  --name AUGAN_MVP_01 \
  --checkpoints_dir /mnt/g/train_data/checkpoints0303 \
  --dir_lq Recon_HQ_33 \
  --dir_sq Recon_SQ_75 \
  --batch_size 2 \
  --patch_size_d 128 \
  --patch_size_h 64 \
  --patch_size_w 64 \
  --lr 0.0002 \
  --n_epochs 100 \
  --gpu_ids 0

✅ Epoch [1/100] 结束 | 耗时: 119.48 秒 | 最新 L1: 0.0727                               
✅ Epoch [2/100] 结束 | 耗时: 112.34 秒 | 最新 L1: 0.0460                               
✅ Epoch [3/100] 结束 | 耗时: 112.23 秒 | 最新 L1: 0.0812                               
✅ Epoch [4/100] 结束 | 耗时: 103.47 秒 | 最新 L1: 0.0605                               
✅ Epoch [5/100] 结束 | 耗时: 109.45 秒 | 最新 L1: 0.0672                               
✅ Epoch [6/100] 结束 | 耗时: 103.59 秒 | 最新 L1: 0.0526                               
Epoch [7/100]:  85%|▊| 127/150 [01:31<00:16,  1.39it/s, Loss_D=0.1943, Loss_G=0.^C
Exception ignored in: <function ArrayProxy.__del__ at 0x7d29bebd9510>
Traceback (most recent call last):
  File "/home/liujia/miniconda3/envs/augan_wsl/lib/python3.10/site-packages/nibabel/arrayproxy.py", line 237, in __del__
    def __del__(self):
KeyboardInterrupt: 
Exception ignored in: <function ArrayProxy.__del__ at 0x7d29bebd9510>
Traceback (most recent call last):
  File "/home/liujia/miniconda3/envs/augan_wsl/lib/python

In [26]:
!python test.py \
  --dataroot /home/liujia/g_linux/test/Simu_3_33_75/ \
  --name AUGAN_MVP_01 \
  --checkpoints_dir /mnt/g/train_data/checkpoints0303 \
  --dir_lq Recon_HQ_33 \
  --epoch 5 \
  --gpu_ids 0

>>> 正在加载权重: /mnt/g/train_data/checkpoints0303/AUGAN_MVP_01/netG_epoch_5.pth
Testing: 100%|█| 3/3 [00:39<00:00, 13.07s/it, File=SimData_NII_0003_Pts_192_hq_3

>>> 测试圆满完成！所有 NIfTI 和三视图已保存至: /mnt/g/train_data/checkpoints0303/AUGAN_MVP_01/test_results_epoch_5
